# 05 · ETL — Geodades: Assentaments i Outposts (capes de mapa)

**Fonts:**
- `peacenow_settlements_geo.csv` — output del notebook 01 (148 assentaments amb coordenades)
- `peacenow_outposts_geo.csv` — output del notebook 01 (383 outposts amb coordenades)

**Outputs** (a `data/clean/`):
- `settlements_points.csv` — capa de punts d'assentaments, llesta per al mapa (capa 6)
- `outposts_points.csv` — capa de punts d'outposts, llesta per al mapa (capa 7)
- `settlements_outposts_points.csv` — les dues capes combinades amb columna `type`, per a un toggle únic

---
## Rol en l'arquitectura

Aquest notebook alimenta les capes 6 i 7 del mapa interactiu (notebook 06):
assentaments i outposts com a marcadors puntuals.

---
## ⚠️ Pendent — NO cobert per aquest notebook

Aquestes capes de la narrativa territorial **no es poden generar** perquè encara no hi ha
cap font raw amb geometria a `Datasets/raw/`. No s'inventa ni s'aproxima cap dada:

| Capa narrativa | Per què no es pot fer ara | Font que caldria |
|---|---|---|
| 2. Green Line | Cap geometria de línia disponible (només `dist_green_line_km`, que és una distància, no la línia) | Shapefile/GeoJSON de la Green Line (p. ex. OCHA oPt) |
| 3. Àrees A/B/C | Cap fitxer de polígons | Shapefile/GeoJSON d'àrees A/B/C (OCHA oPt) |
| 4. Barrera | Cap fitxer de geometria de la barrera | Shapefile/GeoJSON de la barrera (OCHA oPt) |
| 5. Checkpoints | Cap fitxer de punts | Shapefile/GeoJSON/CSV de checkpoints (OCHA oPt) |
| 8. Demolicions (geolocalitzades) | `demolitions_wb.csv` només té `locality`/`district` en text, sense coordenades | Gasetter de localitats palestines amb lat/lon per fer el join per nom |
| 9. Víctimes (geolocalitzades) | `fatalities_settlers_wb.csv` té el mateix problema | Mateix gasetter que a demolicions |

Quan tinguis aquestes fonts, es poden afegir com a seccions noves d'aquest mateix notebook
(o com a 05b si prefereixes separar-ho) sense tocar el que ja funciona aquí.


## 1. Importació de llibreries

In [1]:
import pandas as pd
import os

print("Llibreries carregades correctament")

Llibreries carregades correctament


## 2. Configuració de paths

Aquest notebook llegeix outputs ja nets del notebook 01 (no dades raw).

In [2]:
# Els inputs són outputs del notebook 01, ja a data/clean/
CLEAN = "data/clean"

FILE_SETTLEMENTS = f"{CLEAN}/peacenow_settlements_geo.csv"
FILE_OUTPOSTS    = f"{CLEAN}/peacenow_outposts_geo.csv"

OUT_SETTLEMENTS = f"{CLEAN}/settlements_points.csv"
OUT_OUTPOSTS     = f"{CLEAN}/outposts_points.csv"
OUT_COMBINED     = f"{CLEAN}/settlements_outposts_points.csv"

os.makedirs(CLEAN, exist_ok=True)

print("Paths configurats:")
print(f"  CLEAN: {os.path.abspath(CLEAN)}")

Paths configurats:
  CLEAN: c:\Users\a-iba\OneDrive\Documentos\Sprint13\Notebook\data\clean


---
## Secció A — Capa de punts: Assentaments

148 assentaments. 1 sense coordenades (Tel Zion, escindit de Kochav Yaakov el 2023) — es documenta i s'exclou de la capa de mapa.


In [3]:
df_settlements = pd.read_csv(FILE_SETTLEMENTS)
print(f"Shape original: {df_settlements.shape}")

sense_coords = df_settlements[df_settlements["lat"].isna()]
print(f"\nSense coordenades ({len(sense_coords)}):")
print(sense_coords["settlement"].tolist())

Shape original: (148, 13)

Sense coordenades (1):
['Tel Zion']


In [4]:
# Excloure files sense coordenades (no es poden posicionar al mapa)
df_settlements_map = df_settlements[df_settlements["lat"].notna()].copy()

# Estandarditzar per a la capa de mapa
df_settlements_map = df_settlements_map.rename(columns={"settlement": "name"})
df_settlements_map["type"] = "Settlement"

# Columnes rellevants per al mapa (es descarten x_itm/y_itm, ja redundants amb lat/lon)
df_settlements_map = df_settlements_map[[
    "name", "name_hebrew", "db_id", "type",
    "lat", "lon", "year_established",
    "dist_green_line_km", "municipality", "urban_pattern", "elevation_m"
]].rename(columns={"urban_pattern": "category"})

print(f"Shape capa assentaments: {df_settlements_map.shape}")
print(f"\nRang lat/lon (comprovació de coherència geogràfica, WB aprox. 31.3-32.6 / 34.9-35.6):")
print(f"  lat: {df_settlements_map['lat'].min():.2f} - {df_settlements_map['lat'].max():.2f}")
print(f"  lon: {df_settlements_map['lon'].min():.2f} - {df_settlements_map['lon'].max():.2f}")

Shape capa assentaments: (147, 11)

Rang lat/lon (comprovació de coherència geogràfica, WB aprox. 31.3-32.6 / 34.9-35.6):
  lat: 31.36 - 32.48
  lon: 34.90 - 35.53


---
## Secció B — Capa de punts: Outposts

383 outposts, tots amb coordenades. Normalitzem `settlement_type` (inconsistència de majúscules a la font).


In [5]:
df_outposts = pd.read_csv(FILE_OUTPOSTS)
print(f"Shape original: {df_outposts.shape}")
print(f"Sense coordenades: {df_outposts['lat'].isna().sum()}")
print(f"\nsettlement_type abans de normalitzar:")
print(df_outposts["settlement_type"].value_counts())

Shape original: (383, 12)
Sense coordenades: 0

settlement_type abans de normalitzar:
settlement_type
Farm Outpost              195
Outpost                   160
Farm outpost               27
Farm Outpost - Yeshiva      1
Name: count, dtype: int64


In [6]:
df_outposts_map = df_outposts.copy()

# Normalitzar capitalització inconsistent ("Farm outpost" vs "Farm Outpost")
df_outposts_map["settlement_type"] = df_outposts_map["settlement_type"].str.title()

df_outposts_map["type"] = "Outpost"

df_outposts_map = df_outposts_map[[
    "name", "name_hebrew", "db_id", "type",
    "lat", "lon", "year_established",
    "district", "municipality", "settlement_type", "nearest_settlement"
]].rename(columns={"settlement_type": "category"})

print(f"Shape capa outposts: {df_outposts_map.shape}")
print(f"\nsettlement_type desprès de normalitzar:")
print(df_outposts_map["category"].value_counts())

Shape capa outposts: (383, 11)

settlement_type desprès de normalitzar:
category
Farm Outpost              222
Outpost                   160
Farm Outpost - Yeshiva      1
Name: count, dtype: int64


---
## Secció C — Capa combinada (assentaments + outposts)

Un únic fitxer amb columna `type` per activar/desactivar ambdues capes juntes al mapa si cal.
Les columnes que no apliquen a un tipus queden buides (NaN) per a l'altre.


In [7]:
df_points = pd.concat([df_settlements_map, df_outposts_map], ignore_index=True, sort=False)

print(f"Shape capa combinada: {df_points.shape}")
print(f"\nPer tipus:")
print(df_points["type"].value_counts())
print(f"\nNuls per columna:")
print(df_points.isnull().sum())

Shape capa combinada: (530, 13)

Per tipus:
type
Outpost       383
Settlement    147
Name: count, dtype: int64

Nuls per columna:
name                    0
name_hebrew             0
db_id                  33
type                    0
lat                     0
lon                     0
year_established        7
dist_green_line_km    393
municipality           11
category               10
elevation_m           408
district              179
nearest_settlement    169
dtype: int64


## 3. Exportació

In [8]:
# A — Assentaments
df_settlements_map.to_csv(OUT_SETTLEMENTS, index=False)
print(f"✓ {OUT_SETTLEMENTS}")
print(f"  {len(df_settlements_map)} punts")

# B — Outposts
df_outposts_map.to_csv(OUT_OUTPOSTS, index=False)
print(f"\n✓ {OUT_OUTPOSTS}")
print(f"  {len(df_outposts_map)} punts")

# C — Combinada
df_points.to_csv(OUT_COMBINED, index=False)
print(f"\n✓ {OUT_COMBINED}")
print(f"  {len(df_points)} punts totals")

print(f"\nCapes 6 i 7 (assentaments, outposts) llestes per al notebook 06.")
print(f"Capes 2-5 (Green Line, Àrees A/B/C, barrera, checkpoints) i 8-9 (demolicions,")
print(f"víctimes geolocalitzades) pendents de fonts raw — veure nota a l'inici del notebook.")

✓ data/clean/settlements_points.csv
  147 punts

✓ data/clean/outposts_points.csv
  383 punts

✓ data/clean/settlements_outposts_points.csv
  530 punts totals

Capes 6 i 7 (assentaments, outposts) llestes per al notebook 06.
Capes 2-5 (Green Line, Àrees A/B/C, barrera, checkpoints) i 8-9 (demolicions,
víctimes geolocalitzades) pendents de fonts raw — veure nota a l'inici del notebook.
